In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv(r"C:\Users\panwa\Downloads\archive\ecommerce_customer_data_large.csv")
df.head()

In [ ]:
df.columns

In [ ]:
df = df.drop(columns=['Customer Name'])

In [ ]:
df['Age'] = df['Age'].fillna(df['Customer Age'])
df = df.drop(columns=['Customer Age'])

In [ ]:
df['Purchase Date'] = pd.to_datetime(df['Purchase Date'])

In [ ]:
df.isnull().sum()
df.fillna(0, inplace=True)

In [ ]:
df['Total Purchase Amount'] = df['Product Price'] * df['Quantity']

In [ ]:
##Create Customer Summary


In [ ]:
customer_df = df.groupby('Customer ID').agg({
    'Total Purchase Amount': 'sum',
    'Quantity': 'sum',
    'Customer ID': 'count',
    'Returns': 'sum',
    'Churn': 'first',
    'Age': 'first',
    'Gender': 'first'
})

customer_df.rename(columns={
    'Customer ID': 'Frequency'
}, inplace=True)

customer_df.head()

In [ ]:
#RFM Analysis

snapshot_date = df['Purchase Date'].max()

rfm = df.groupby('Customer ID').agg({
    'Purchase Date': lambda x: (snapshot_date - x.max()).days,
    'Customer ID': 'count',
    'Total Purchase Amount': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

In [ ]:
rfm.groupby('Cluster').mean()

In [ ]:
#category sales
df.groupby('Product Category')['Total Purchase Amount'].sum().sort_values(ascending=False)

In [ ]:
##payment method

df['Payment Method'].value_counts()

In [ ]:
#Monthly sales trend
monthly = df.resample('M', on='Purchase Date')['Total Purchase Amount'].sum()
monthly.plot()
plt.title("Monthly Sales Trend")
plt.show()

In [ ]:
#RETURN & CHURN ANALYSIS

df['Returns'].value_counts()

In [ ]:
df['Churn'].value_counts()

In [ ]:
import seaborn as sns
sns.boxplot(x='Churn', y='Total Purchase Amount', data=df)
plt.show()

In [ ]:
sns.scatterplot(x='Frequency', y='Monetary', hue='Cluster', data=rfm)
plt.show()

In [ ]:
df.groupby('Product Category')['Total Purchase Amount'].sum().plot(kind='bar')
plt.show()

In [ ]:
sns.barplot(x='Gender', y='Total Purchase Amount', data=df)
plt.show()

In [ ]:
at_risk = rfm[rfm['Recency'] > 60]

In [ ]:
df

In [ ]:
monthly_sales = df.groupby(
    df['Purchase Date'].dt.to_period('M')
)['Total Purchase Amount'].sum()

monthly_sales.plot(title='Monthly Purchase Trend')
plt.show()

In [ ]:
rfm['R_score'] = pd.qcut(rfm['Recency'],4,labels=[4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['Frequency'],4,labels=[1,2,3,4])
rfm['M_score'] = pd.qcut(rfm['Monetary'],4,labels=[1,2,3,4])

rfm['RFM_score'] = (
    rfm['R_score'].astype(int) +
    rfm['F_score'].astype(int) +
    rfm['M_score'].astype(int)
)

rfm.head()

In [ ]:
def segment_customer(score):
    
    if score >= 10:
        return "High-Value"
    
    elif score >= 6:
        return "Mid-Value"
    
    else:
        return "Low-Value"

In [ ]:
rfm['Segment'] = rfm['RFM_score'].apply(segment_customer)

rfm.head()

In [ ]:
rfm['Segment'].value_counts().plot(
    kind='bar',
    title='Customer Segment Distribution'
)

plt.show()

In [ ]:
df = df.merge(
    rfm[['Segment']],
    left_on='Customer ID',
    right_index=True,
    how='left'
)